# Autonomous Graph of Thoughts (GoT) Debugging Agent
### Quantized Coding LLM (Qwen2.5-Coder-7B / Llama-3-8B) + Code Property Graph in Google Colab

This notebook runs an autonomous LLM debugging agent powered by **Code Property Graph (CPG)** static analysis.

**Core Mechanism: Graph of Thoughts (GoT)**
- The model **cannot see the whole codebase**.
- Given a failing test symptom, it forms hypotheses, invokes graph traversal tools (`traverse_ast`, `traverse_cfg`, `traverse_data_flow`), merges observations across syntax, execution paths, and data dependencies, and pinpoints the exact bug and patch.

## Step 1: Clone Repository directly into Colab
Set your GitHub repository URL below. Colab will clone it and enter the repository folder.

In [ ]:
import os

# Set your repository URL here:
REPO_URL = "https://github.com/nour-ezzehi/GDAC.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_NAME) and not os.path.exists("agent_runner.py"):
    print(f"[*] Cloning {REPO_URL}...")
    !git clone $REPO_URL
    %cd $REPO_NAME
elif os.path.exists(REPO_NAME):
    %cd $REPO_NAME
    print("[*] Pulling latest changes...")
    !git pull
else:
    print("[+] Already inside repository directory.")

!pwd && ls -la

## Step 2: Install GPU Packages
Install PyTorch, HuggingFace Transformers, BitsAndBytes (for 4-bit quantization), Accelerate, and NetworkX.

In [ ]:
!nvidia-smi
!pip install -q torch transformers bitsandbytes accelerate networkx

## Step 3: Quick Test & Graph Tool Verification (Dry-Run)
Verify that the CPG graph files in `cpg_json/` and the three graph traversal tools load properly.

In [ ]:
# Runs a local simulation verifying tool dispatch on the loaded CPG graph
!python agent_runner.py --cpg-path cpg_json --dry-run --target discount

## Step 4: Load 4-bit Quantized Model (Qwen2.5-Coder-7B or Llama-3-8B)
Loads the model into GPU VRAM using 4-bit NormalFloat (NF4) quantization (~5.5 GB VRAM), which runs comfortably on Colab's standard T4 GPU.

In [ ]:
from agent_runner import load_quantized_model

# Loads Qwen2.5-Coder-7B-Instruct with 4-bit quantization
model, tokenizer = load_quantized_model(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    quantization="4bit"
)

## Step 5: Execute Autonomous Graph of Thoughts Debugging
The agent will:
1. Read the test failure report.
2. Formulate an initial hypothesis.
3. Call `traverse_ast` to locate the target method and understand its parameters.
4. Call `traverse_cfg` to trace runtime branches leading to failure.
5. Call `traverse_data_flow` to trace variable mutations and calculations.
6. Output the exact root cause and code patch.

### Bug Scenario 1: Discount Calculation Defect (`apply_discount`)

In [ ]:
from cpg_bridge import load_cpg_json
from agent_prompt import format_user_problem_prompt
from agent_runner import run_agent_loop

graph = load_cpg_json("cpg_json")

problem_prompt_1 = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_inventory.py::TestInventoryManager::test_apply_discount\n"
        "AssertionError: 20.0 != 80.0\n"
        "Applying a 20% discount on $100 item resulted in $20.00 instead of $80.00."
    ),
    bug_description="Unit test fails when calculating percentage discounts."
)

deduction_1 = run_agent_loop(model, tokenizer, graph, problem_prompt_1, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(deduction_1)

### Bug Scenario 2: Category Summary Aggregation Defect (`generate_category_summary`)

In [ ]:
problem_prompt_2 = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_reports.py::TestInventoryReporter::test_generate_category_summary_aggregates_multiple_items\n"
        "AssertionError: 250.0 != 1000.0\n"
        "Category summary only shows the last product value ($250) instead of aggregate total ($1000)."
    ),
    bug_description="Category valuation summary ignores multiple items in the same category."
)

deduction_2 = run_agent_loop(model, tokenizer, graph, problem_prompt_2, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(deduction_2)